# Inventar Projects to Hubs Linker

Links infrastructure inventory projects to transit hubs via H3 hexagonal spatial intersection.

**Pipeline Position**: This module enriches output from the Hub Processing Pipeline (Parts 1-3)
by identifying which planned infrastructure projects intersect with each hub's H3 cells.

**Key Features**:
- H3 hexagon to Shapely polygon conversion
- R-tree spatial indexing for fast intersection queries
- Project data joining with filtering capabilities
- Excludes road projects (main_type='כביש') from analysis

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuration

In [16]:
# =============================================================================
# CONFIGURATION - Edit these paths for your environment
# =============================================================================

# Working directory
WD = r'/content/drive/MyDrive/Hubs_Urgency'

# Input files
HUBS_CSV = WD + r'/Data/Results_29-12-2025.csv'  # Hub results from Pipeline Parts 1-3
INVENTAR_DIR = WD + r'/Data/Inventar'            # Directory with shapefiles
PROJECT_DATA_CSV = WD + r'/Data/Inventar/data.csv'        # Project metadata with main_type column

# Output files
OUTPUT_CSV = WD + r'/Hubs_w_InventarProjects.csv'           # Base output
OUTPUT_COMBINED_CSV = WD + r'/Hubs_w_InventarProjects_combined.csv'  # With all_project_uids
OUTPUT_EXPLODED_CSV = WD + r'/Hubs_w_InventarProjects_exploded.csv'  # One row per project
OUTPUT_FILTERED_CSV = WD + r'/Hubs_w_InventarProjects_filtered.csv'  # Excluding roads

# Encoding for Hebrew text
ENCODING = 'windows-1255'

## 2. Install Dependencies

In [17]:
# Ensure correct h3 version
!pip uninstall h3 -y && pip install h3 -q
!pip install geopandas shapely -q

Found existing installation: h3 4.4.1
Uninstalling h3-4.4.1:
  Successfully uninstalled h3-4.4.1


## 3. Imports and Setup

In [18]:
import pandas as pd
import geopandas as gpd
import numpy as np
import h3
import logging
from pathlib import Path
from shapely.geometry import Polygon
from typing import List, Any, Optional, Set

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"h3 version: {h3.__version__}")
print(f"geopandas version: {gpd.__version__}")

h3 version: 4.4.1
geopandas version: 1.1.1


## 4. Helper Functions

In [19]:
def parse_h3_list(x) -> list:
    """
    Parse H3 index list from string representation.

    Handles formats:
    - "['abc123', 'def456']"
    - "[abc123, def456]"
    - Already parsed lists
    """
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    # Remove brackets and split
    return [item.strip().strip("'\"")
            for item in str(x).replace('[', '').replace(']', '').split(', ')
            if item.strip()]


def h3_to_polygon(h3_index_str: str) -> Optional[Polygon]:
    """
    Convert H3 index to Shapely Polygon.

    H3 returns (lat, lon), Shapely expects (lon, lat).
    """
    if not isinstance(h3_index_str, str):
        return None
    try:
        geo_boundary = h3.cell_to_boundary(h3_index_str)
        # Swap (lat, lon) to (lon, lat) for Shapely
        lon_lat_boundary = [(lon, lat) for lat, lon in geo_boundary]
        return Polygon(lon_lat_boundary)
    except Exception as e:
        logger.warning(f"Error converting H3 {h3_index_str}: {e}")
        return None


def find_intersecting_features(
    polygon: Polygon,
    gdf: gpd.GeoDataFrame,
    uid_column: str = 'uid'
) -> List[Any]:
    """
    Find all features in GeoDataFrame that intersect with polygon.
    Uses spatial index for performance.
    """
    if polygon is None or gdf.empty:
        return []

    try:
        # Stage 1: Bounding box filter via spatial index
        candidate_indices = list(gdf.sindex.intersection(polygon.bounds))
        if not candidate_indices:
            return []

        # Stage 2: Precise intersection test
        candidates = gdf.iloc[candidate_indices]
        intersecting = candidates[candidates.intersects(polygon)]

        return intersecting[uid_column].tolist()
    except Exception as e:
        logger.warning(f"Intersection error: {e}")
        return []

## 5. Load Hub Data

In [20]:
# Load hub results
logger.info(f"Loading hub data from {HUBS_CSV}")
hub_df = pd.read_csv(HUBS_CSV, encoding=ENCODING)

print(f"Loaded {len(hub_df)} hub groups")
print(f"Columns: {list(hub_df.columns)}")
hub_df.head()

Loaded 135 hub groups
Columns: ['group', 'x', 'y', 'h3_index', 'node', 'Mode_Planned', 'Modes_ForPlot', 'Line_Unique', 'Line_Names', 'Line_Names_forPlot', 'address', 'area', 'Metro', 'location', 'LocationForChart', 'TotalDemand', 'TotalTransfers', 'TransferRate', 'BRT Lines', 'Cable Line Lines', 'Funicular Lines', 'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines', 'Metro Lines', 'Suburban Rail Lines', 'pop_0_500', 'emp_0_500', 'pop_500_1000', 'emp_500_1000', 'pop_1000_1500', 'emp_1000_1500', 'TotalPop_2050', 'TotalEmp_2050', 'Region_category', 'Location_category', 'RegionLocation', 'Num_Modes', 'score', 'bus_terminal', 'HubType', 'HubType_Filtered', 'HubTypeHE', 'BusTERMINAL_Clone', 'RegionLocation_Norm', 'score_Norm', 'bus_terminal_Norm', 'TotalDemand_Norm', 'PopEmp_Score_Norm', 'TotalScore_MC', 'Rank_TS_MC', 'Rank_By_TS_MC_By_Metro', 'TotalNumLines', 'NumLinesStatus_0', 'NumLinesStatus_1', 'NumLinesStatus_2', 'NumLinesStatus_3', 'NumLinesStatus_4', 'NumLinesStatus_5', 'Num

,group,x,y,h3_index,node,Mode_Planned,Modes_ForPlot,Line_Unique,Line_Names,Line_Names_forPlot,...,Average_Simulated_Score,Overall_Rank,Rank_within_HubType,LogDemand,PopEmp_Score,HubNameHE,TotalScore_Manual,Rank_TS_Manual,Rank_TS_Manual_By_Metro,Rank_By_LogDemand
0,528,35.108145,32.816776,"['8a2db0a52227fff', '8a2db0a523affff']","[34010, 1003]","['BRT', 'LRT']","BRT, רק""ל","[""['lrt03', 'lrt04', 'lrt02', 'lrt01']"", ""['R1...","['נצרת-חוף הכרמל (רק""ל)', 'חוף הכרמל-נצרת (רק""...","נצרת-חוף הכרמל (רק""ל), חוף הכרמל-נצרת (רק""ל), ...",...,2.900458,91,20,3.096683,26777.991780,אצטדיון קרית אתא,1.737661,35,10,35
1,115,34.936611,32.220850,['8a2db018a967fff'],[511153],"['BRT', 'LRT']","BRT, רק""ל","[""['LRT142', 'LRT141']""]","['כפ""ס-נתניה דרך טייבה/קלאנסווה (רק""ל)', 'נתני...","כפ""ס-נתניה דרך טייבה/קלאנסווה (רק""ל), נתניה-כפ...",...,1.971824,129,27,3.097968,2180.763650,צומת רמת הכובש,1.008008,27,27,27
2,23,34.989450,32.240650,['8a2db00a5b67fff'],[511148],"['BRT', 'LRT']","BRT, רק""ל","[""['LRT142', 'LRT141']""]","['כפ""ס-נתניה דרך טייבה/קלאנסווה (רק""ל)', 'נתני...","כפ""ס-נתניה דרך טייבה/קלאנסווה (רק""ל), נתניה-כפ...",...,2.407221,114,24,3.208648,4493.943819,צומת צור נתן,1.347255,26,26,26
3,104,34.839784,32.214399,"['8a2db018496ffff', '8a2db01a34a7fff']","[400040, 511248]","['Interurban Rail', 'Suburban Rail', 'LRT']","רכבת בינעירונית, רכבת פרברית, רק""ל","[""['LRT132', 'LRT122', 'LRT121', 'LRT131']"", ""...","['הרצליה-נתניה מזרח (רק""ל)', 'הרצליה-נתניה מער...","הרצליה-נתניה מזרח (רק""ל), הרצליה-נתניה מערב (ר...",...,4.094823,41,10,3.219969,1417.847498,חנה וסע שפיים,2.681065,30,21,34
4,32,34.875043,32.318872,['8a2db01187b7fff'],[511260],"['BRT', 'LRT']","BRT, רק""ל","[""['LRT142', 'LRT141']""]","['כפ""ס-נתניה דרך טייבה/קלאנסווה (רק""ל)', 'נתני...","כפ""ס-נתניה דרך טייבה/קלאנסווה (רק""ל), נתניה-כפ...",...,2.746931,98,21,3.338339,21284.753620,"פנקס/האורג, נתניה",1.838921,24,24,25


In [21]:
# Parse H3 index lists
hub_df['h3_index'] = hub_df['h3_index'].apply(parse_h3_list)

# Select relevant columns
columns_to_keep = ['group', 'x', 'y', 'HubNameHE', 'h3_index']
available_columns = [c for c in columns_to_keep if c in hub_df.columns]
hub_df = hub_df[available_columns]

# Explode: one row per H3 cell
hub_df = hub_df.explode('h3_index').reset_index(drop=True)

print(f"Exploded to {len(hub_df)} H3 cells")
hub_df.head()

Exploded to 264 H3 cells


,group,x,y,HubNameHE,h3_index
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a523affff
2,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff
3,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff
4,104,34.839784,32.214399,חנה וסע שפיים,8a2db018496ffff


## 6. Load Inventar Shapefiles

In [22]:
inventar_path = Path(INVENTAR_DIR)

# Load shapefiles
geometry_layers = {}

shapefile_map = {
    'points': 'geom_point.shp',
    'lines': 'geom_line.shp',
    'multilines': 'geom_multiline.shp'
}

for name, filename in shapefile_map.items():
    filepath = inventar_path / filename
    if filepath.exists():
        gdf = gpd.read_file(filepath)
        geometry_layers[name] = gdf
        print(f"✓ Loaded {name}: {len(gdf)} features")
    else:
        geometry_layers[name] = gpd.GeoDataFrame()
        print(f"✗ Not found: {filepath}")

✓ Loaded points: 152 features
✓ Loaded lines: 1037 features
✓ Loaded multilines: 223 features


In [23]:
# Preview each layer
for name, gdf in geometry_layers.items():
    if len(gdf) > 0:
        print(f"\n{name.upper()}:")
        display(gdf.head(2))


POINTS:


,uid,geometry
0,2018,MULTIPOINT ((35.19752 31.80261))
1,2020,MULTIPOINT ((35.22801 31.86123))



LINES:


,uid,geometry
0,4,"LINESTRING (34.89929 32.48427, 34.89927 32.484..."
1,5,"LINESTRING (35.04367 32.38911, 35.04365 32.389..."



MULTILINES:


,uid,geometry
0,302,"MULTILINESTRING ((34.96328 31.90832, 34.96329 ..."
1,306,"MULTILINESTRING ((34.6339 31.78727, 34.63447 3..."


## 7. Convert H3 to Polygons

In [24]:
logger.info("Converting H3 indices to polygons...")
hub_df['h3_polygon_geom'] = hub_df['h3_index'].apply(h3_to_polygon)

# Check conversion success
valid_polygons = hub_df['h3_polygon_geom'].notna().sum()
print(f"Successfully converted {valid_polygons}/{len(hub_df)} H3 cells to polygons")
hub_df.head()

Successfully converted 264/264 H3 cells to polygons


,group,x,y,HubNameHE,h3_index,h3_polygon_geom
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,..."
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a523affff,"POLYGON ((35.10708856325309 32.81642219935896,..."
2,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff,"POLYGON ((34.936142519104 32.221471361317015, ..."
3,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,..."
4,104,34.839784,32.214399,חנה וסע שפיים,8a2db018496ffff,POLYGON ((34.83873325888983 32.214040485217275...


## 8. Compute Spatial Intersections

In [25]:
logger.info("Computing spatial intersections...")

# Initialize result lists
all_intersecting_points = []
all_intersecting_lines = []
all_intersecting_multilines = []

total_rows = len(hub_df)
log_interval = max(1, total_rows // 10)

for idx, row in hub_df.iterrows():
    polygon = row['h3_polygon_geom']

    # Find intersections for each geometry type
    all_intersecting_points.append(
        find_intersecting_features(polygon, geometry_layers['points'])
    )
    all_intersecting_lines.append(
        find_intersecting_features(polygon, geometry_layers['lines'])
    )
    all_intersecting_multilines.append(
        find_intersecting_features(polygon, geometry_layers['multilines'])
    )

    # Progress logging
    if (idx + 1) % log_interval == 0:
        logger.info(f"Processed {idx + 1}/{total_rows} rows ({100*(idx+1)//total_rows}%)")

# Assign results
hub_df['intersecting_points'] = all_intersecting_points
hub_df['intersecting_lines'] = all_intersecting_lines
hub_df['intersecting_multilines'] = all_intersecting_multilines

logger.info("Intersection computation complete!")

# Summary statistics
has_points = hub_df['intersecting_points'].apply(len).gt(0).sum()
has_lines = hub_df['intersecting_lines'].apply(len).gt(0).sum()
has_multilines = hub_df['intersecting_multilines'].apply(len).gt(0).sum()

print(f"\n=== Intersection Summary ===")
print(f"H3 cells with point intersections: {has_points}")
print(f"H3 cells with line intersections: {has_lines}")
print(f"H3 cells with multiline intersections: {has_multilines}")


=== Intersection Summary ===
H3 cells with point intersections: 6
H3 cells with line intersections: 208
H3 cells with multiline intersections: 4


## 9. Combine and Explode UIDs

In [26]:
def combine_project_uids(df: pd.DataFrame) -> pd.DataFrame:
    """
    Combine all intersecting UIDs into a single 'all_project_uids' column.
    """
    result = df.copy()

    def merge_uids(row):
        all_uids = set()
        for col in ['intersecting_points', 'intersecting_lines', 'intersecting_multilines']:
            if col in row and isinstance(row[col], list):
                all_uids.update(row[col])
        return list(all_uids)

    result['all_project_uids'] = result.apply(merge_uids, axis=1)
    return result


def explode_by_project(df: pd.DataFrame) -> pd.DataFrame:
    """
    Explode dataframe so each row has exactly one project_uid.
    """
    if 'all_project_uids' not in df.columns:
        df = combine_project_uids(df)

    # Explode on all_project_uids
    exploded = df.explode('all_project_uids').reset_index(drop=True)
    exploded = exploded.rename(columns={'all_project_uids': 'project_uid'})

    # Remove rows with no project
    exploded = exploded[exploded['project_uid'].notna()]

    return exploded

In [27]:
# Create combined version
hub_combined = combine_project_uids(hub_df)
print(f"Combined DataFrame: {len(hub_combined)} rows")
print(f"Total unique projects found: {hub_combined['all_project_uids'].apply(len).sum()}")

hub_combined.head()

Combined DataFrame: 264 rows
Total unique projects found: 437


,group,x,y,HubNameHE,h3_index,h3_polygon_geom,intersecting_points,intersecting_lines,intersecting_multilines,all_project_uids
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],"[200009, 7042]"
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a523affff,"POLYGON ((35.10708856325309 32.81642219935896,...",[],[],[],[]
2,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff,"POLYGON ((34.936142519104 32.221471361317015, ...",[],[408004],[],[408004]
3,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],"[20106, 408004]"
4,104,34.839784,32.214399,חנה וסע שפיים,8a2db018496ffff,POLYGON ((34.83873325888983 32.214040485217275...,[],"[32003, 200017, 408002, 408001]",[],"[200017, 408002, 32003, 408001]"


In [28]:
# Create exploded version (one row per project_uid)
hub_exploded = explode_by_project(hub_combined)
print(f"Exploded DataFrame: {len(hub_exploded)} rows")
print(f"Unique project UIDs: {hub_exploded['project_uid'].nunique()}")

hub_exploded.head()

Exploded DataFrame: 437 rows
Unique project UIDs: 133


,group,x,y,HubNameHE,h3_index,h3_polygon_geom,intersecting_points,intersecting_lines,intersecting_multilines,project_uid
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],200009
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],7042
3,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff,"POLYGON ((34.936142519104 32.221471361317015, ...",[],[408004],[],408004
4,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],20106
5,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],408004


## 10. Load Project Data and Filter Roads

Join project metadata from `data.csv` and filter out road projects (`main_type='כביש'`).

In [29]:
# Load project metadata
logger.info(f"Loading project data from {PROJECT_DATA_CSV}")

try:
    project_df = pd.read_csv(PROJECT_DATA_CSV, encoding=ENCODING)
except UnicodeDecodeError:
    project_df = pd.read_csv(PROJECT_DATA_CSV, encoding='utf-8-sig')

print(f"Loaded {len(project_df)} projects")
print(f"\nColumns: {list(project_df.columns)[:15]}...")  # Show first 15 columns

# Show main_type distribution
print(f"\nmain_type distribution:")
print(project_df['main_type'].value_counts())

Loaded 1412 projects

Columns: ['uid', 'proj_name', 'district', 'road', 'road_other', 'auth_name', 'street', 'start', 'finish', 'fromKm', 'toKm', 'main_type', 'is_iu', 'geometry_type', 'proj_type']...

main_type distribution:
main_type
כביש           1237
מתע"ן            88
רכבת             72
מתקנים ואחר      15
Name: count, dtype: int64


In [30]:
# Select relevant columns for joining
project_cols = ['uid', 'proj_name', 'main_type', 'Proj_status', 'scn_year']
available_project_cols = [c for c in project_cols if c in project_df.columns]
project_subset = project_df[available_project_cols].copy()

print(f"Project subset columns: {available_project_cols}")
project_subset.head()

Project subset columns: ['uid', 'proj_name', 'main_type', 'Proj_status', 'scn_year']


,uid,proj_name,main_type,Proj_status,scn_year
0,4,תכנית מתאר כוללנית למרחב קסריה,כביש,?,2040
1,5,כביש דרומי לג'ת,כביש,?,2040
2,6,כביש מערבי לבאקה-ג'ת,כביש,?,2040
3,7,חיבור בין 6513 ל- 6524 בקציר,כביש,?,מעבר ל-2050
4,8,חיבור כביש 6513 לכביש 6524 בקציר,כביש,?,מעבר ל-2050


In [31]:
class ProjectDataFilter:
    """
    Handles joining and filtering of project data.

    Single Responsibility: Filter operations on project data.
    """

    def __init__(self, project_df: pd.DataFrame):
        self.project_df = project_df.copy()
        self._prepare_join_key()

    def _prepare_join_key(self):
        """Ensure uid column is proper type for joining."""
        if 'uid' in self.project_df.columns:
            # Convert to int if possible, keep as-is otherwise
            self.project_df['uid'] = pd.to_numeric(
                self.project_df['uid'], errors='coerce'
            )

    def join_to_hubs(self, hub_df: pd.DataFrame, hub_uid_col: str = 'project_uid') -> pd.DataFrame:
        """
        Join project data to hub dataframe.

        Args:
            hub_df: Exploded hub dataframe with project_uid column
            hub_uid_col: Column name containing project UIDs in hub_df

        Returns:
            Joined dataframe
        """
        result = hub_df.copy()

        # Ensure matching types
        result[hub_uid_col] = pd.to_numeric(result[hub_uid_col], errors='coerce')

        # Perform join
        joined = result.merge(
            self.project_df,
            left_on=hub_uid_col,
            right_on='uid',
            how='left'
        )

        logger.info(f"Joined {len(joined)} rows (from {len(result)} hub rows)")
        return joined

    def filter_exclude_roads(self, df: pd.DataFrame, main_type_col: str = 'main_type') -> pd.DataFrame:
        """
        Filter out road projects (main_type='כביש').

        Args:
            df: DataFrame with main_type column
            main_type_col: Column name for main_type

        Returns:
            Filtered dataframe excluding roads
        """
        if main_type_col not in df.columns:
            logger.warning(f"Column '{main_type_col}' not found. Returning unfiltered data.")
            return df

        before_count = len(df)

        # Filter out roads (כביש)
        filtered = df[df[main_type_col] != 'כביש'].copy()

        after_count = len(filtered)
        removed = before_count - after_count

        logger.info(f"Filtered out {removed} road projects (כביש). {after_count} rows remaining.")

        return filtered

    def get_excluded_uids(self, main_type_value: str = 'כביש') -> Set[int]:
        """
        Get set of UIDs that should be excluded based on main_type.

        Useful for filtering before joining.
        """
        mask = self.project_df['main_type'] == main_type_value
        return set(self.project_df.loc[mask, 'uid'].dropna().astype(int))

In [32]:
# Initialize filter
project_filter = ProjectDataFilter(project_subset)

# Get UIDs to exclude (roads)
road_uids = project_filter.get_excluded_uids('כביש')
print(f"Road project UIDs to exclude: {len(road_uids)}")

Road project UIDs to exclude: 1237


In [33]:
# Join project data to exploded hubs
hub_with_projects = project_filter.join_to_hubs(hub_exploded, hub_uid_col='project_uid')

print(f"\nJoined DataFrame shape: {hub_with_projects.shape}")
print(f"\nmain_type distribution in joined data:")
print(hub_with_projects['main_type'].value_counts())

hub_with_projects.head()


Joined DataFrame shape: (437, 15)

main_type distribution in joined data:
main_type
מתע"ן    285
כביש     114
רכבת      38
Name: count, dtype: int64


,group,x,y,HubNameHE,h3_index,h3_polygon_geom,intersecting_points,intersecting_lines,intersecting_multilines,project_uid,uid,proj_name,main_type,Proj_status,scn_year
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],200009,200009,מסילת זבולון,רכבת,?,2050
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],7042,7042,"רק""ל נופית","מתע""ן",ביצוע,2030
2,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff,"POLYGON ((34.936142519104 32.221471361317015, ...",[],[408004],[],408004,408004,LRT14 - BRT - קאלאנסווה - טירה - כפר סבא,"מתע""ן",תכנון ראשוני,2040
3,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],20106,20106,כביש 444 - הרחבה (הוספת נתיב מנוהל) - מצומת שמ...,כביש,תכנון מוקדם,2030
4,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],408004,408004,LRT14 - BRT - קאלאנסווה - טירה - כפר סבא,"מתע""ן",תכנון ראשוני,2040


In [34]:
# Filter out road projects
hub_filtered = project_filter.filter_exclude_roads(hub_with_projects)

print(f"\nFiltered DataFrame shape: {hub_filtered.shape}")
print(f"\nmain_type distribution after filtering:")
print(hub_filtered['main_type'].value_counts())

hub_filtered.head()


Filtered DataFrame shape: (323, 15)

main_type distribution after filtering:
main_type
מתע"ן    285
רכבת      38
Name: count, dtype: int64


,group,x,y,HubNameHE,h3_index,h3_polygon_geom,intersecting_points,intersecting_lines,intersecting_multilines,project_uid,uid,proj_name,main_type,Proj_status,scn_year
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],200009,200009,מסילת זבולון,רכבת,?,2050
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],7042,7042,"רק""ל נופית","מתע""ן",ביצוע,2030
2,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff,"POLYGON ((34.936142519104 32.221471361317015, ...",[],[408004],[],408004,408004,LRT14 - BRT - קאלאנסווה - טירה - כפר סבא,"מתע""ן",תכנון ראשוני,2040
4,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],408004,408004,LRT14 - BRT - קאלאנסווה - טירה - כפר סבא,"מתע""ן",תכנון ראשוני,2040
5,104,34.839784,32.214399,חנה וסע שפיים,8a2db018496ffff,POLYGON ((34.83873325888983 32.214040485217275...,[],"[32003, 200017, 408002, 408001]",[],200017,200017,מסילות 5 6 באיילון,רכבת,?,2050


## 11. Summary Statistics

In [35]:
print("="*60)
print("PROCESSING SUMMARY")
print("="*60)
print(f"\nInput:")
print(f"  - Hub groups: {hub_df['group'].nunique() if 'group' in hub_df.columns else 'N/A'}")
print(f"  - H3 cells: {len(hub_df)}")
print(f"  - Projects in data.csv: {len(project_df)}")

print(f"\nSpatial Intersections:")
print(f"  - H3 cells with point projects: {has_points}")
print(f"  - H3 cells with line projects: {has_lines}")
print(f"  - H3 cells with multiline projects: {has_multilines}")

print(f"\nAfter Exploding:")
print(f"  - Total hub-project pairs: {len(hub_exploded)}")
print(f"  - Unique projects intersecting hubs: {hub_exploded['project_uid'].nunique()}")

print(f"\nAfter Filtering (excluding כביש):")
print(f"  - Hub-project pairs: {len(hub_filtered)}")
print(f"  - Unique non-road projects: {hub_filtered['project_uid'].nunique()}")
print(f"  - Removed road entries: {len(hub_with_projects) - len(hub_filtered)}")

print(f"\nProject Types in Final Output:")
print(hub_filtered['main_type'].value_counts().to_string())

PROCESSING SUMMARY

Input:
  - Hub groups: 135
  - H3 cells: 264
  - Projects in data.csv: 1412

Spatial Intersections:
  - H3 cells with point projects: 6
  - H3 cells with line projects: 208
  - H3 cells with multiline projects: 4

After Exploding:
  - Total hub-project pairs: 437
  - Unique projects intersecting hubs: 133

After Filtering (excluding כביש):
  - Hub-project pairs: 323
  - Unique non-road projects: 72
  - Removed road entries: 114

Project Types in Final Output:
main_type
מתע"ן    285
רכבת      38


## 12. Save Results

In [36]:
# Prepare dataframes for export (drop geometry column)
export_columns = [c for c in hub_df.columns if c != 'h3_polygon_geom']

# Save base output (with intersection lists)
hub_df[export_columns].to_csv(OUTPUT_CSV, index=False, encoding=ENCODING)
print(f"✓ Saved base output: {OUTPUT_CSV}")

# Save combined output (with all_project_uids)
combined_export_cols = [c for c in hub_combined.columns if c != 'h3_polygon_geom']
hub_combined[combined_export_cols].to_csv(OUTPUT_COMBINED_CSV, index=False, encoding=ENCODING)
print(f"✓ Saved combined output: {OUTPUT_COMBINED_CSV}")

# Save exploded output (one row per project)
exploded_export_cols = [c for c in hub_exploded.columns if c != 'h3_polygon_geom']
hub_exploded[exploded_export_cols].to_csv(OUTPUT_EXPLODED_CSV, index=False, encoding=ENCODING)
print(f"✓ Saved exploded output: {OUTPUT_EXPLODED_CSV}")

# Save filtered output (excluding roads, with project metadata)
filtered_export_cols = [c for c in hub_filtered.columns if c != 'h3_polygon_geom']
hub_filtered[filtered_export_cols].to_csv(OUTPUT_FILTERED_CSV, index=False, encoding=ENCODING)
print(f"✓ Saved filtered output (no roads): {OUTPUT_FILTERED_CSV}")

✓ Saved base output: /content/drive/MyDrive/Hubs_Urgency/Hubs_w_InventarProjects.csv
✓ Saved combined output: /content/drive/MyDrive/Hubs_Urgency/Hubs_w_InventarProjects_combined.csv
✓ Saved exploded output: /content/drive/MyDrive/Hubs_Urgency/Hubs_w_InventarProjects_exploded.csv
✓ Saved filtered output (no roads): /content/drive/MyDrive/Hubs_Urgency/Hubs_w_InventarProjects_filtered.csv


In [39]:
hub_filtered

,group,x,y,HubNameHE,h3_index,h3_polygon_geom,intersecting_points,intersecting_lines,intersecting_multilines,project_uid,uid,proj_name,main_type,Proj_status,scn_year
0,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],200009,200009,מסילת זבולון,רכבת,?,2050
1,528,35.108145,32.816776,אצטדיון קרית אתא,8a2db0a52227fff,"POLYGON ((35.10826272042373 32.81837377977052,...",[],"[7042, 200009]",[],7042,7042,"רק""ל נופית","מתע""ן",ביצוע,2030
2,115,34.936611,32.220850,צומת רמת הכובש,8a2db018a967fff,"POLYGON ((34.936142519104 32.221471361317015, ...",[],[408004],[],408004,408004,LRT14 - BRT - קאלאנסווה - טירה - כפר סבא,"מתע""ן",תכנון ראשוני,2040
4,23,34.989450,32.240650,צומת צור נתן,8a2db00a5b67fff,"POLYGON ((34.98898203905635 32.24127203943853,...",[],"[20106, 408004]",[],408004,408004,LRT14 - BRT - קאלאנסווה - טירה - כפר סבא,"מתע""ן",תכנון ראשוני,2040
5,104,34.839784,32.214399,חנה וסע שפיים,8a2db018496ffff,POLYGON ((34.83873325888983 32.214040485217275...,[],"[32003, 200017, 408002, 408001]",[],200017,200017,מסילות 5 6 באיילון,רכבת,?,2050
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426,643,34.797273,32.083390,תחנת רכבת סבידור מרכז,8a2db0cceceffff,"POLYGON ((34.7981290443205 32.08342285253187, ...",[],"[7016, 32003, 200017]",[],7016,7016,"הקו האדום - רק""ל - זרוע מרכזית","מתע""ן",?,2025
427,643,34.797273,32.083390,תחנת רכבת סבידור מרכז,8a2db0cceceffff,"POLYGON ((34.7981290443205 32.08342285253187, ...",[],"[7016, 32003, 200017]",[],200017,200017,מסילות 5 6 באיילון,רכבת,?,2050
429,643,34.797273,32.083390,תחנת רכבת סבידור מרכז,8a2db0ccec0ffff,"POLYGON ((34.79556154064352 32.08332511608337,...",[],[7027],[],7027,7027,m1 - מטרו - גזע מרכזי,"מתע""ן",תכנון מפורט,2040
433,592,34.794529,32.072633,תחנת רכבת השלום,8a2db0cc3217fff,"POLYGON ((34.79497042767569 32.07243643380726,...",[],"[32028, 37017, 7030]",[],7030,7030,m2 - מטרו,"מתע""ן",תכנון מפורט,2040


## 13. Optional: Visualization

In [ ]:
# Uncomment to create interactive map
# !pip install folium -q

# import folium
#
# # Center map on data
# map_center = [hub_df['y'].mean(), hub_df['x'].mean()]
# m = folium.Map(location=map_center, zoom_start=10)
#
# # Add H3 cells with projects
# for idx, row in hub_combined[hub_combined['all_project_uids'].apply(len) > 0].head(100).iterrows():
#     polygon = row['h3_polygon_geom']
#     if polygon:
#         coords = [(lat, lon) for lon, lat in polygon.exterior.coords]
#         folium.Polygon(
#             locations=coords,
#             color='blue',
#             fill=True,
#             fill_opacity=0.4,
#             tooltip=f"Group: {row.get('group', 'N/A')}<br>Projects: {len(row['all_project_uids'])}"
#         ).add_to(m)
#
# m

---

## Summary

This notebook:

1. Loaded hub data from the Transit Hub Processing Pipeline
2. Loaded Inventar geometry shapefiles (points, lines, multilines)
3. Converted H3 hexagonal indices to Shapely polygons
4. Computed spatial intersections using R-tree spatial indexing
5. Combined all intersecting UIDs into a single column
6. Exploded to create one row per hub-project pair
7. Joined project metadata from data.csv
8. **Filtered out road projects (main_type='כביש')**
9. Saved multiple output formats for different use cases

### Output Files:

| File | Description | Use Case |
|------|-------------|----------|
| `*_base.csv` | One row per H3 cell, lists of UIDs | Detailed spatial analysis |
| `*_combined.csv` | Same + `all_project_uids` column | Combined view |
| `*_exploded.csv` | One row per (H3, project) pair | SQL joins, pivot tables |
| `*_filtered.csv` | Exploded + joined + **no roads** | Transit-focused analysis |